# Benchmark `string-grouperx` vs `string_grouper`

Compares 4 configurations on the SEC EDGAR dataset (~663k company names), matmul backend `sp_matmul_rs` everywhere except the baseline row:

1. **upstream sparse_dot_topn** — `string_grouper`, old matmul backend (historical baseline)
2. **upstream sp_matmul_rs** — `string_grouper` with the PR #105 backend
3. **sgx sklearn** — `string_grouperx`, Python vectorizer (isolates the gain *outside* the vectorizer)
4. **sgx rust** — `string_grouperx`, Rust vectorizer (full config)

Method: best-of-N on wall time (a single run is inflated by system noise). We also report total CPU time and **effective parallelism** = CPU/wall, which shows *why* it is faster (filling the single-core gap of vectorization).

In [1]:
# --- Config ---
COMPANY_NAMES = '/Users/guillaumepressiat/Downloads/sec_edgar_company_info.txt'
N_REPEATS = 3          # repetitions per config (best-of-N)
MIN_SIMILARITY = 0.8   # threshold; keep the string_grouper default

import os, time, platform
import pandas as pd
import polars as pl
import string_grouper as sg
import string_grouperx as sgx

companies_pd = pd.read_csv(COMPANY_NAMES)
companies_pl = pl.read_csv(COMPANY_NAMES)
names_pd = companies_pd['Company Name']
names_pl = companies_pl.select('Company Name').to_series()
N = len(names_pd)
print(f'{N:,} company names | {os.cpu_count()} logical CPUs | {platform.processor() or platform.machine()}')

663,000 company names | 10 logical CPUs | i386


In [2]:
def bench(fn, n_repeats=N_REPEATS):
    """Return (best_wall, cpu_at_best, n_rows) over n_repeats runs.
    CPU measured via time.process_time() (all threads of the process)."""
    best = None
    for _ in range(n_repeats):
        t0 = time.perf_counter(); c0 = time.process_time()
        out = fn()
        wall = time.perf_counter() - t0; cpu = time.process_time() - c0
        n_rows = out.shape[0] if hasattr(out, 'shape') else len(out)
        if best is None or wall < best[0]:
            best = (wall, cpu, n_rows)
    return best

CONFIGS = {
    'upstream sparse_dot_topn': lambda: sg.match_strings(names_pd, min_similarity=MIN_SIMILARITY, use_sp_matmul_rs=False),
    'upstream sp_matmul_rs':    lambda: sg.match_strings(names_pd, min_similarity=MIN_SIMILARITY, use_sp_matmul_rs=True),
    'sgx sklearn':              lambda: sgx.match_strings(names_pl, min_similarity=MIN_SIMILARITY, vectorizer='sklearn'),
    'sgx rust':                 lambda: sgx.match_strings(names_pl, min_similarity=MIN_SIMILARITY, vectorizer='rust'),
}

In [3]:
results = []
for label, fn in CONFIGS.items():
    print(f'  {label} ...', end=' ', flush=True)
    wall, cpu, n_rows = bench(fn)
    print(f'{wall:.1f}s')
    results.append({'config': label, 'wall_s': wall, 'cpu_s': cpu, 'rows': n_rows})

  upstream sparse_dot_topn ... 

2026-07-20 21:56:30.504 | INFO     | string_grouper.string_grouper:_calc_blocks_and_build_matches:431 - n_blocks parameter is not set so data will be split into smaller chunks, n_blocks = (1,166)
2026-07-20 21:57:35.522 | INFO     | string_grouper.string_grouper:_calc_blocks_and_build_matches:431 - n_blocks parameter is not set so data will be split into smaller chunks, n_blocks = (1,166)
2026-07-20 21:58:41.229 | INFO     | string_grouper.string_grouper:_calc_blocks_and_build_matches:431 - n_blocks parameter is not set so data will be split into smaller chunks, n_blocks = (1,166)


65.1s
37.1stream sp_matmul_rs ... 
32.9s sklearn ... 
26.8s rust ... 


In [5]:
df = pd.DataFrame(results)

# relative speeds: baseline = upstream sp_matmul_rs (same matmul backend as sgx)
base = df.loc[df.config == 'upstream sp_matmul_rs', 'wall_s'].iloc[0]
df['speedup_vs_sp_matmul'] = base / df['wall_s']
df['parallelism'] = df['cpu_s'] / df['wall_s']

def fmt_time(s):
    return f'{int(s // 60)}min{s % 60:04.1f}s' if s >= 60 else f'{s:.1f}s'

view = pd.DataFrame({
    'Configuration': df['config'],
    'Wall (best of %d)' % N_REPEATS: df['wall_s'].map(fmt_time),
    'Total CPU': df['cpu_s'].map(fmt_time),
    'Parallelism (CPU/wall)': df['parallelism'].map(lambda x: f'{x:.1f}x'),
    'Speedup vs sp_matmul_rs': df['speedup_vs_sp_matmul'].map(lambda x: f'{x:.2f}x'),
    'Rows': df['rows'].map(lambda n: f'{n:,}'),
})

styled = (view.style
    .hide(axis='index')
    .set_caption(f'SEC EDGAR — {N:,} company names, {os.cpu_count()} CPUs, min_similarity={MIN_SIMILARITY}')
    .set_table_styles([
        {'selector': 'caption', 'props': [('font-weight', 'bold'), ('font-size', '13px'), ('padding-bottom', '8px')]},
        {'selector': 'th', 'props': [('background-color', '#1f2937'), ('color', 'white'), ('text-align', 'left'), ('padding', '6px 10px')]},
        {'selector': 'td', 'props': [('padding', '6px 10px'), ('border-bottom', '1px solid #e5e7eb')]},
    ])
    .apply(lambda col: ['background-color: #ecfdf5; font-weight: bold' if v == 'sgx rust' else '' for v in df['config']], axis=0)
)
styled

Configuration,Wall (best of 3),Total CPU,Parallelism (CPU/wall),Speedup vs sp_matmul_rs,Rows
upstream sparse_dot_topn,1min05.1s,7min21.6s,6.8x,0.57x,"1,594,336"
upstream sp_matmul_rs,37.1s,3min39.3s,5.9x,1.00x,"1,594,336"
sgx sklearn,32.9s,3min32.9s,6.5x,1.13x,"1,594,336"
sgx rust,26.8s,3min27.3s,7.7x,1.38x,"1,594,336"


In [6]:
# Gain decomposition (at equal matmul backend)
w = df.set_index('config')['wall_s']
up = w['upstream sp_matmul_rs']; sk = w['sgx sklearn']; rs = w['sgx rust']
print('Decomposition, matmul backend sp_matmul_rs held constant:')
print(f'  upstream                                : {up:.1f}s')
print(f'  + pandas-free rewrite (sgx sklearn)     : {up:.1f} -> {sk:.1f}s  ({(1-sk/up)*100:+.0f}%)')
print(f'  + Rust vectorizer     (sgx rust)        : {sk:.1f} -> {rs:.1f}s  ({(1-rs/sk)*100:+.0f}%)')
print(f'  = total gain                            : {up:.1f} -> {rs:.1f}s  (x{up/rs:.2f})')
print(f'\nNote: total CPU time varies little; the gain comes from parallelism (single-core gap filled).')

Decomposition, matmul backend sp_matmul_rs held constant:
  upstream                                : 37.1s
  + pandas-free rewrite (sgx sklearn)     : 37.1 -> 32.9s  (+11%)
  + Rust vectorizer     (sgx rust)        : 32.9 -> 26.8s  (+18%)
  = total gain                            : 37.1 -> 26.8s  (x1.38)

Note: total CPU time varies little; the gain comes from parallelism (single-core gap filled).


In [2]:
# Parity (validated method: non-saturated max_n_matches, sort, floating-point tolerance)
import numpy as np
key = ['left_index', 'right_index']
ref = (sg.match_strings(names_pd, min_similarity=MIN_SIMILARITY, max_n_matches=10000)
         .reset_index(drop=True).sort_values(key).reset_index(drop=True))
out = (sgx.match_strings(names_pl, min_similarity=MIN_SIMILARITY, max_n_matches=10000)
         .to_pandas().sort_values(key).reset_index(drop=True))
same_pairs = ref[key].equals(out[key])
sim_diff = (ref['similarity'].to_numpy() - out['similarity'].to_numpy())
print(f'same pairs (index)  : {same_pairs}')
print(f'max similarity diff : {np.abs(sim_diff).max():.2e}')
print(f'rows : {len(ref):,} vs {len(out):,}')

same pairs (index)  : True
max similarity diff : 1.22e-15
rows : 3,356,972 vs 3,356,972
